In [1]:
import pandas as pd
import numpy as np

# Читаем с оптимизированными типами сразу при загрузке
dtype_map = {
    "id":             "category",
    "item_id":        "category",
    "dept_id":        "category",
    "cat_id":         "category",
    "store_id":       "category",
    "state_id":       "category",
    "d":              "category",
    "weekday":        "category",
    "event_name_1":   "category",
    "event_type_1":   "category",
    "event_name_2":   "category",
    "snap_CA":        np.int8,
    "snap_TX":        np.int8,
    "snap_WI":        np.int8,
    "sales":          np.int32,
    "wday":           np.int8,
    "month":          np.int8,
    "year":           np.int16,
    "week_of_year":   np.int16,
    "day_of_year":    np.int16,
    "d_num":          np.int16,
    "age_of_series":  np.int16,
    "is_weekend":     np.int8,
    "has_event":      np.int8,
    "is_top_event":   np.int8,
    "is_promo":       np.int8,
    "is_new_sku":     np.int8,
    "is_outlier_day": np.int8,
    "sell_price":          np.float32,
    "price_roll_mean_28":  np.float32,
    "price_rel":           np.float32,
    "price_change":        np.float32,
    "lag_1":               np.float32,
    "lag_2":               np.float32,
    "lag_7":               np.float32,
    "lag_14":              np.float32,
    "lag_28":              np.float32,
    "lag_56":              np.float32,
    "roll_mean_7":         np.float32,
    "roll_std_7":          np.float32,
    "roll_mean_28":        np.float32,
    "roll_std_28":         np.float32,
    "roll_mean_56":        np.float32,
    "roll_std_56":         np.float32,
}

print("Загружаем датасет...")
df = pd.read_csv("m5_dataset_final.csv", dtype=dtype_map, parse_dates=["date"])

print(f"Shape: {df.shape}")
print(f"Память: {df.memory_usage(deep=True).sum() / 1024**3:.2f} GB")
print(f"\nПропуски:")
nulls = df.isnull().sum()
print(nulls[nulls > 0] if nulls[nulls > 0].any() else "Пропусков нет ✅")

Загружаем датасет...
Shape: (33202011, 47)
Память: 6.34 GB

Пропуски:
event_type_2          33125272
sell_price             5326353
price_roll_mean_28     5326353
price_rel              5326353
lag_1                    30490
lag_2                    60978
lag_7                   213360
lag_14                  426464
lag_28                  851539
lag_56                 1695345
roll_mean_7              30490
roll_mean_28             30490
roll_mean_56             30490
dtype: int64


In [2]:
# 1. event_type_2 — почти пустая, дропаем
df = df.drop(columns=["event_type_2"])
print("event_type_2 удалён ✅")

# 2. sell_price — ffill/bfill внутри SKU
df["sell_price"] = (
    df.groupby("id", observed=True)["sell_price"]
    .transform(lambda x: x.ffill().bfill())
).astype(np.float32)

# Пересчитываем зависимые ценовые признаки
df["price_roll_mean_28"] = (
    df.groupby("id", observed=True)["sell_price"]
    .transform(lambda x: x.rolling(28, min_periods=1).mean())
).astype(np.float32)

df["price_rel"] = (df["sell_price"] / df["price_roll_mean_28"]).astype(np.float32)
df["is_promo"]  = (df["price_rel"] < 0.9).astype(np.int8)
print("Ценовые признаки пересчитаны ✅")

# 3. Лаги — заполняем -1 (нет истории ≠ продаж 0)
lag_cols  = [f"lag_{l}" for l in [1, 2, 7, 14, 28, 56]]
roll_cols = [f"roll_mean_{w}" for w in [7, 28, 56]]
df[lag_cols + roll_cols] = df[lag_cols + roll_cols].fillna(-1).astype(np.float32)
print("Лаги заполнены ✅")

# 4. Финальная проверка
print(f"\nShape: {df.shape}")
print(f"Память: {df.memory_usage(deep=True).sum() / 1024**3:.2f} GB")
nulls = df.isnull().sum()
print(f"\nПропуски:")
print(nulls[nulls > 0] if nulls[nulls > 0].any() else "Пропусков нет ✅")

# 5. Сохраняем в parquet
print("\nСохранение в parquet...")
df.to_parquet("m5_dataset_final.parquet", index=False)
print(f"Готово: m5_dataset_final.parquet")

event_type_2 удалён ✅
Ценовые признаки пересчитаны ✅
Лаги заполнены ✅

Shape: (33202011, 46)
Память: 5.35 GB

Пропуски:
Пропусков нет ✅

Сохранение в parquet...
Готово: m5_dataset_final.parquet
